In [ ]:
from itertools import combinations,permutations,product,chain
from operator import itemgetter
from math import comb,factorial

"""
Signed Partial Matching between [n] and [m]
Is a map from subset of [n] to subset of [m] with sign +1 or -1
"""

def setsign(a, sign):
    assert(len(a)==len(sign))
    return tuple(a if s == 1 else -a for a,s in zip(a,sign))

def SPM(m,n,k):
    assert(k <= min(n,m))
    #generate all subset of [n] with size k
    for domain in combinations(range(1,m+1), k):
        for codomain in permutations(range(1,n+1), k):
            for sign in product((1,-1), repeat=k):
                yield tuple(zip(domain,codomain,sign))

"""
s_i=(i,i+1) in type A subroot system of W_1 
s_{-i}=(i,-i) in type B subroot system of W_1
"""
def sact(i,sigma):
    if i>0: 
        R = tuple(sorted(((i+1, s[1],s[2]) if s[0]==i 
                 else ((i,s[1],s[2]) if s[0]==i+1
                 else s) for s in sigma),key=itemgetter(0)))
    elif i < 0:
        R = tuple((s[0],s[1],-1*s[2]) if s[0]== -i else s for s in sigma)
    else:
        raise ValueError("i must be non-zero")
    return R 

def sactp(i,sigma):
    if i>0: 
        R = tuple((s[0],i+1,s[2]) if s[1]==i 
                 else ((s[0],i,s[2]) if s[1]==i+1
                 else s) for s in sigma)
    elif i < 0:
        R = tuple((s[0],s[1],-1*s[2]) if s[1]== -i else s for s in sigma)
    else:
        raise ValueError("i must be non-zero")
    return R 

def SPMgen(m,n,k):
    """
    The generator of SPM(m,n,k) 
    """
    assert(k <= min(n,m))
    I = range(1,k+1) 
    J = range(1,k+1)
    return tuple(zip(I,J,(1,)*k))


def SPMgen(m,n,k):
    I = range(m+1-k,m+1) 
    J = range(1,k+1)
    return tuple(zip(I,J,(1,)*k))



def simplereflectionA(m):
    return [i for i in range(1,m+1)]

def simplerefA(m):
    for i in range(1,m):
        yield tuple(chain(range(1,i), (i+1,i),range(i+2,m+1)))

def simplerefB(m):
    yield tuple(chain(range(1,m),(-m,)))
    for i in range(1,m):
        yield tuple(chain(range(1,i), (i+1,i),range(i+2,m+1)))

def simplereflectionB(m):
    return [i for i in range(1,m+1)]+[-m]

def naivelenght1(m,n,k):
    """
    figure out the length layer by layer
    """
    A = set(SPM(m,n,k))
    gen = SPMgen(m,n,k)
    A.remove(gen)
    D = dict() 
    D[gen] = 0
    flag = False
    Level = 0
    while len(A)>0:
        Keys = [k for k,v in D.items() if v == Level]
        Level += 1    
        for sigma in Keys:
            for i in simplereflectionA(m): 
                R = sact(i,sigma)
                if R in A:
                    D[R] = Level
                    A.remove(R)
            for j in simplereflectionB(n):
                R = sactp(j,sigma)
                if R in A:
                    D[R] = Level
                    A.remove(R)
        #print(len(A))
    return D         

In [ ]:
def multperm(perm1,perm2):
    """
    Multiply two signed permutations
    """
    return tuple(perm2[i-1] if i>0 else -perm2[-i-1] for i in perm1)

def lengthA(perm): 
    """
    A permutation is a ordered list of positive integers.
    length is the inversion number of the permutation.
    """
    return sum(1 for i in range(len(perm)) for j in range(i+1, len(perm)) if perm[i] > perm[j])

def lengthB(perm):
    """
    A permutation is a ordered list of integers.
    length is the inversion number of the permutation.
    
    We use the formula (8.2) of [BB, Coxeter]
    """
    w0 = tuple(range(len(perm),0,-1))
    perm = multperm(multperm(w0,perm),w0)
    
    return sum(1 for i in range(len(perm)) for j in range(i+1, len(perm)) if perm[i] > perm[j]) \
        + sum(1 for i in range(len(perm)) for j in range(i, len(perm)) if - perm[i] > perm[j])


def lengthD(perm):
    """
    A signed permutation is a ordered list of integers.
    This function calculate extended type D Coxeter group. 
    We use the formula (8.18) of [BB, Coxeter]
    """
    w0 = tuple(range(len(perm),0,-1))
    perm = multperm(multperm(w0,perm),w0)
    
    return sum(1 for i in range(len(perm)) for j in range(i+1, len(perm)) if perm[i] > perm[j]) \
        + sum(1 for i in range(len(perm)) for j in range(i+1, len(perm)) if - perm[i] > perm[j])

def leftcosetA(n,k):
    """
    Generate the minimal length representative of S_n / S_(n-k) x S_k 
    """
    assert(k<=n)
    A = list(range(1,n+1))
    for r in combinations(A,n-k):
        yield tuple(chain(r,(a for a in A if a not in r)))

def leftcosetB(n,k):
    """
    Generate the minimal length representative of W_n / 1 x W_(n-k) 
    """
    assert(k<=n)
    A = list(range(1,n+1))
    for r in combinations(A,k):
        rc = tuple(a for a in A if a not in r)
        for r in permutations(r):
            for sign in product((1,-1),repeat=k):
                yield tuple(chain(setsign(r,sign),rc))


def reg_sigma(sigma):
    """
    Regularity of a sigma
    """
    return tuple(sorted(sigma,key=itemgetter(0))) 

def Aact(perm,sigma):
    """
    Apply the perm action of W1 on sigma 
    """
    nsigma = reg_sigma((perm[s[0]-1],s[1],s[2]) if perm[s[0]-1]>0 else 
                   (-perm[s[0]-1],s[1],-s[2]) for s in sigma)
    return nsigma

def Bact(perm,sigma):
    """
    Apply the perm action of W2 on sigma 
    """
    nsigma = tuple((s[0],perm[s[1]-1],s[2]) if perm[s[1]-1]>0 else 
                   (s[0],-perm[s[1]-1],-s[2]) for s in sigma)
    return nsigma

def cleverlength(m,n,k):
    D = dict()
    gen = SPMgen(m,n,k)
    for p1 in leftcosetA(m,k):
        for p2 in leftcosetB(n,k):
            nsigma = gen
            nsigma = Aact(p1,nsigma)
            nsigma = Bact(p2,nsigma)
            l = lengthA(p1) + lengthB(p2)
            #print(f"sigma: {nsigma}")
            #print(f"{p1}:{lengthA(p1)}, {p2}:{lengthB(p2)}")
            D[nsigma] = l
    return D

def cleverlengthp(m,n,k):
    D = dict()
    gen = SPMgen(n,m,k)
    for p1 in leftcosetA(n,k):
        for p2 in leftcosetB(m,k):
            nsigma = gen
            nsigma = Aact(p1,nsigma)
            nsigma = Bact(p2,nsigma)
            l = lengthA(p1) + lengthD(p2)
            #print(f"sigma: {nsigma}")
            #print(f"{p1}:{lengthA(p1)}, {p2}:{lengthB(p2)}")
            D[nsigma] = l
    return D

def sizemnk(m,n,k):
    return comb(m,k)*comb(n,k)*(2**k)*factorial(k)

def naivelenght(m,n,k):
    """
    figure out the length layer by layer
    """
    A = set(SPM(m,n,k))
    gen = SPMgen(m,n,k)
    A.remove(gen)
    D = dict() 
    D[gen] = 0
    while len(A)>0:
        KV = []
        for sigma in A:
            l = None
            for p1 in simplerefA(m): 
                R = Aact(p1,sigma)
                if R in D and (l==None or D[R]<l):
                    l = D[R]+1
            for p2 in simplerefB(n):
                R = Bact(p2,sigma)
                if R in D and (l==None or D[R]<l):
                    l = D[R]+1
            KV.append((sigma,l))
        for sigma,l in KV:
            if l != None:
                D[sigma] = l
                A.remove(sigma)
    return D

def compairedict(D1,D2):
    """
    Compare the two dictionaries
    """
    isequal = True
    for k in D1:
        if D1[k]!=D2[k]:
            print(f"Key: {k}, {D1[k]}, {D2[k]}")
            isequal = False
    return isequal

def SPMheart(m,n,k):
    """
    The heart of the SPM algorithm
    """
    D = cleverlength(m,n,k)
    SPMH = []
    for sigma, v in D.items():
        inheart = True
        for p1 in simplerefA(m):
            R = Aact(p1,sigma)
            #print(f"sigma:{sigma}, p1:{p1}, R:{R}")
            if D[R]<D[sigma]:
                inheart = False
                break
        for p2 in simplerefA(n):
            R = Bact(p2,sigma)
            if D[R]<D[sigma]:
                inheart = False
                break
        if inheart:
            SPMH.append(sigma)
    return SPMH            

def SPMH(m,n):
    """
    The SPM heart for all k 
    """
    H = []
    for k in range(min(m,n)+1):
        print(f"k={k}")
        H.extend(SPMheart(m,n,k))
    return H



def SPMheartp(m,n,k):
    """
    The heart of the SPM algorithm
    """
    D = cleverlengthp(m,n,k)
    SPMH = []
    for sigma, v in D.items():
        inheart = True
        for p1 in simplerefA(n):
            R = Aact(p1,sigma)
            #print(f"sigma:{sigma}, p1:{p1}, R:{R}")
            if D[R]<D[sigma]:
                inheart = False
                break
        for p2 in simplerefA(m):
            R = Bact(p2,sigma)
            if D[R]<D[sigma]:
                inheart = False
                break
        if inheart:
            SPMH.append(sigma)
    return SPMH            

def SPMHp(m,n):
    """
    The SPM heart for all k 
    """
    H = []
    for k in range(min(m,n)+1):
        print(f"k={k}")
        H.extend(SPMheartp(m,n,k))
    return H

def sizeheart(m,n):
    return sum(1 for i in range(m+1) for j in range(n+1) if i+j<=min(m,n))

In [ ]:
m, n, k = 4,7,3
#print(list(SPM(m,n,k)))


A = set(SPM(m,n,k))
D = naivelenght(m,n,k)
Dp = naivelenght1(m,n,k)
Dpp = cleverlength(m,n,k)
assert(A == set(D.keys()))

print(f"{sizemnk(m,n,k)}={len(set(D.keys()))}")
print(f"{len(set(A))}={len(set(D.keys()))}")
print(f"{len(set(A))}={len(set(Dp.keys()))}")
print(f"{len(set(A))}={len(set(Dpp.keys()))}")

assert(A == set(Dpp.keys()))

print(f"{SPMgen(m,n,k)}")

In [ ]:
print(max(D.values()))
print(max(Dp.values()))
print(max(Dpp.values()))
compairedict(D,Dpp)
assert(set(D.items()) == set(Dp.items()))
D.keys()


In [ ]:
H1 = SPMH(m,n)
print(f"{len(H1)}, {H1}")
H2 = SPMHp(m,n)
print(f"{len(H2)}, {H2}")
print(f"SPMH size: {sizeheart(m,n)}")